In [7]:
!pip install opencv-python
!pip install numpy
!pip install scikit-learn
!pip install keras
!pip install tensorflow
!apt-get update && apt-get install -y libgl1
!pip install mtcnn
!pip install keras_facenet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 43.2 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Get:1 http://deb.debian.org/debian bullseye InRelease [116 kB]
Get:2 http://deb.debian.org/debian-security bullseye-security InRelease [27.2 kB]
Get:3 http://deb.debian.org/debian bullseye-updates InRelease [44.1 kB]
Get:4 http://deb.debian.org/debian bullseye/main amd64 Packages [8,066 kB]
Get:5 http://deb.debian.org/debian-security bullseye-security/

In [7]:
import cv2
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, recall_score
from keras.models import load_model
from keras.preprocessing.image import img_to_array
from keras.applications.vgg16 import preprocess_input

# Load pre-trained deep learning model for feature extraction
cnn_model = load_model('/work/vgg16_model.h5')  # Update with your CNN model path

# SVM for Classification
label_encoder = LabelEncoder()
svm_model = SVC(kernel='linear', probability=True)

# Helper function: Preprocess Image
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Error: Image file not found at {image_path}")
    
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (224, 224))  # Assuming VGG Model Input Size
    image_array = img_to_array(image)
    return preprocess_input(np.expand_dims(image_array, axis=0))

# Feature Extraction
def extract_features(image_path):
    processed_image = preprocess_image(image_path)
    return cnn_model.predict(processed_image)

# Enrollment Phase
def enroll_students(student_images, student_ids):
    features = []
    for image_path in student_images:
        features.append(extract_features(image_path).flatten())
    features = np.array(features)
    labels = label_encoder.fit_transform(student_ids)
    svm_model.fit(features, labels)

# Authentication Phase
def authenticate_student(image_path, confidence_threshold=0.6):
    """Authenticates a student image with security confidence threshold check."""
    if not hasattr(svm_model, "support_"):
        print("Authentication error: SVM model is not trained.")
        return None
    features = extract_features(image_path).reshape(1, -1)
    if hasattr(svm_model, "predict_proba"):
        probs = svm_model.predict_proba(features)[0]
        max_prob = np.max(probs)
        best_class = np.argmax(probs)
        if max_prob < confidence_threshold:
            print(f"Authentication failed: Low confidence score ({max_prob:.2f} < {confidence_threshold}).")
            return None
        return label_encoder.inverse_transform([best_class])[0]
    else:
        prediction = svm_model.predict(features)
        return label_encoder.inverse_transform(prediction)[0]

def allocate_seat(recognized_id):
    # Logic to assign seat based on recognized_id
    return f"Seat allocated for student ID: {recognized_id}"

# Real-Time Attendance Tracking
def track_attendance(video_path):
    cap = cv2.VideoCapture(video_path)
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        # Face detection and alignment: You can integrate MTCNN, YOLO, etc.
        faces = detect_faces(frame)  # Placeholder for face detection function
        for face in faces:
            recognized_id = authenticate_student(face)
            print(allocate_seat(recognized_id))

    cap.release()
    cv2.destroyAllWindows()

# Face Detection (Placeholder Implementation)
def detect_faces(frame):
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faces = face_cascade.detectMultiScale(gray_frame, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    return faces  # Coordinates or images for recognized faces

# Example Usage
if __name__ == "__main__":
    #enroll_students(["/work/Student_img/student1.jpg", "/work/Student_img/student2.jpg"], ["ID_001", "ID_002"])

    # Authenticate Example
    #predicted_id = authenticate_student("/work/Student_img/test_student.jpg")
    #print(allocate_seat(predicted_id))

    # Real-time tracking from video
    #track_attendance("/work/exam_hall_video.mp4")

    print("Face Recognition Exam System Loaded and Ready.")

KernelInterrupted: Execution interrupted by the Jupyter kernel.

In [3]:
enroll_students(["/work/Student_img/student1.jpg", "/work/Student_img/student2.jpg"], ["ID_001", "ID_002"])
track_attendance("/work/exam_hall_video.mp4")
predicted_id = authenticate_student("/work/Student_img/test_student.jpg")
print(allocate_seat(predicted_id))
track_attendance("/work/exam_hall_video.mp4")

1/1 [==============================] - 0s 398ms/step
[ WARN:0@8.787] global loadsave.cpp:268 findDecoder imread_('/work/Student_img/test_student.jpg'): can't open/read file: check file path/integrity


FileNotFoundError: Error: Image file not found at /work/Student_img/test_student.jpg

In [13]:
!pip install mtcnn==1.0.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 27.1 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import cv2
import numpy as np
import os
import joblib
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from keras.models import load_model
from keras.preprocessing.image import img_to_array
from keras.applications.vgg16 import preprocess_input
from mtcnn import MTCNN


cnn_model = load_model('/work/vgg16_model.h5')  # Update with your CNN model path

# Load existing models or initialize new ones
svm_model_path = "/work/svm_model.pkl"
label_encoder_path = "/work/label_encoder.pkl"
attendance_file = "/work/exam_attendance.csv"

if os.path.exists(svm_model_path) and os.path.exists(label_encoder_path):
    svm_model = joblib.load(svm_model_path)
    label_encoder = joblib.load(label_encoder_path)
else:
    svm_model = SVC(kernel='linear', probability=True)
    label_encoder = LabelEncoder()

detector = MTCNN()

# Helper function: Preprocess Image
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Error: Image file not found at {image_path}")
    
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (224, 224))  # Assuming VGG Model Input Size
    image_array = img_to_array(image)
    return preprocess_input(np.expand_dims(image_array, axis=0))

# Feature Extraction
def extract_features(image_path):
    processed_image = preprocess_image(image_path)
    return cnn_model.predict(processed_image)

# Enrollment Phase (Train & Save Model)
def enroll_students(student_images, student_ids):
    features = []
    for image_path in student_images:
        features.append(extract_features(image_path).flatten())
    features = np.array(features)
    
    if len(student_ids) > 0:
        labels = label_encoder.fit_transform(student_ids)
        svm_model.fit(features, labels)
        joblib.dump(svm_model, svm_model_path)
        joblib.dump(label_encoder, label_encoder_path)
        print("Student data enrolled successfully!")

# Authentication Phase
def authenticate_student(image_path, confidence_threshold=0.6):
    """Authenticates a student image with security confidence threshold check."""
    if not hasattr(svm_model, "support_"):
        print("Authentication error: SVM model is not trained.")
        return None
    features = extract_features(image_path).reshape(1, -1)
    if hasattr(svm_model, "predict_proba"):
        probs = svm_model.predict_proba(features)[0]
        max_prob = np.max(probs)
        best_class = np.argmax(probs)
        if max_prob < confidence_threshold:
            print(f"Authentication failed: Low confidence score ({max_prob:.2f} < {confidence_threshold}).")
            return None
        return label_encoder.inverse_transform([best_class])[0]
    else:
        prediction = svm_model.predict(features)
        return label_encoder.inverse_transform(prediction)[0]

def allocate_seat(student_id):
    exam_hall = f"R-{int(student_id.split('_')[-1]) % 5 + 1}"  # Example logic
    seat_id = f"S-{int(student_id.split('_')[-1]) % 30 + 1}"   # Example logic

    # Save attendance record
    attendance_status = "Present"
    data = {"Student ID": student_id, "Status": attendance_status, "Exam Hall": exam_hall, "Seat ID": seat_id}
    df = pd.DataFrame([data])
    if os.path.exists(attendance_file):
        df.to_csv(attendance_file, mode='a', header=False, index=False)
    else:
        df.to_csv(attendance_file, mode='w', header=True, index=False)

    return f"Seat Allocated: {exam_hall} - {seat_id} | Attendance Marked: {attendance_status}"

# Real-Time Attendance Tracking
def track_attendance(video_source=0):
    cap = cv2.VideoCapture(video_source)
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        faces = detect_faces(frame)
        for face in faces:
            x, y, w, h = face
            face_img = frame[y:y+h, x:x+w]  # Crop face
            face_img_path = "/work/temp_face.jpg"
            cv2.imwrite(face_img_path, face_img)  # Save temporarily
            
            try:
                recognized_id = authenticate_student(face_img_path)
                print(allocate_seat(recognized_id))
                cv2.putText(frame, f"ID: {recognized_id}", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            except:
                print("Unknown face detected")
                cv2.putText(frame, "Unknown", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

        cv2.imshow("Exam Attendance System", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# Face Detection using MTCNN
def detect_faces(frame):
    faces = detector.detect_faces(frame)
    detected_faces = []
    for face in faces:
        x, y, w, h = face['box']
        detected_faces.append((x, y, w, h))
    return detected_faces

# Example Usage
if __name__ == "__main__":
    print("Enhanced Face Recognition Exam System Loaded and Ready.")


2025-02-26 04:40:15.540945: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-26 04:40:15.552577: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-02-26 04:40:15.652451: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-02-26 04:40:15.652863: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-02-26 04:40:15.653836: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

In [5]:
enroll_students(["/work/Student_img/student1.jpg", "/work/Student_img/student2.jpg"], ["ID_001", "ID_002"])
track_attendance("/work/exam_hall_video.mp4")
predicted_id = authenticate_student("/work/Student_img/student1.jpg")
print(allocate_seat(predicted_id))
track_attendance("/work/exam_hall_video.mp4")

1/1 [==============================] - 1s 585ms/step
Student data enrolled successfully!
1/1 [==============================] - 1s 588ms/step
Seat Allocated: Room-2 - Seat-2 | Attendance Marked: Present


In [7]:
!pip install keras_facenet==0.3.2

  Preparing metadata (setup.py) ... done
  Created wheel for keras_facenet: filename=keras_facenet-0.3.2-py3-none-any.whl size=10368 sha256=5b23f9f1c7d9d44da5051ce4a9e9612fe7e6127e8cf21d7ae450564cf536a8ee
  Stored in directory: /root/.cache/pip/wheels/1d/d8/a9/85cf04ea29321d2afcb82c0caaafdca9195385f9d68cbc7185
Successfully built keras_facenet

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [9]:
import cv2
import numpy as np
import os
import joblib
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from keras.models import load_model
from keras.preprocessing.image import img_to_array
from keras.applications.vgg16 import preprocess_input
from mtcnn import MTCNN
from keras_facenet import FaceNet

# Load pre-trained FaceNet model for high-accuracy face recognition
facenet_model = FaceNet()

# Load existing models or initialize new ones
svm_model_path = "/work/svm_model.pkl"
label_encoder_path = "/work/label_encoder.pkl"
attendance_file = "/work/exam_attendance.csv"

if os.path.exists(svm_model_path) and os.path.exists(label_encoder_path):
    svm_model = joblib.load(svm_model_path)
    label_encoder = joblib.load(label_encoder_path)
else:
    svm_model = SVC(kernel='linear', probability=True)
    label_encoder = LabelEncoder()

detector = MTCNN()

# Helper function: Preprocess Image
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Error: Image file not found at {image_path}")
    
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return image  # FaceNet expects RGB images directly

# Feature Extraction using FaceNet
def extract_features(image_path):
    processed_image = preprocess_image(image_path)
    embeddings = facenet_model.embeddings([processed_image])
    return embeddings[0]

# Enrollment Phase (Train & Save Model)
def enroll_students(student_images, student_ids):
    features = []
    for image_path in student_images:
        features.append(extract_features(image_path))
    features = np.array(features)
    
    if len(student_ids) > 0:
        labels = label_encoder.fit_transform(student_ids)
        svm_model.fit(features, labels)
        joblib.dump(svm_model, svm_model_path)
        joblib.dump(label_encoder, label_encoder_path)
        print("Student data enrolled successfully!")

# Authentication Phase
def authenticate_student(image_path, confidence_threshold=0.6):
    """Authenticates a student image with security confidence threshold check."""
    if not hasattr(svm_model, "support_"):
        print("Authentication error: SVM model is not trained.")
        return None
    features = extract_features(image_path).reshape(1, -1)
    if hasattr(svm_model, "predict_proba"):
        probs = svm_model.predict_proba(features)[0]
        max_prob = np.max(probs)
        best_class = np.argmax(probs)
        if max_prob < confidence_threshold:
            print(f"Authentication failed: Low confidence score ({max_prob:.2f} < {confidence_threshold}).")
            return None
        return label_encoder.inverse_transform([best_class])[0]
    else:
        prediction = svm_model.predict(features)
        return label_encoder.inverse_transform(prediction)[0]

def allocate_seat(student_id, course=None):
    available_seats = [(r, s) for r in range(1, 6) for s in range(1, 31)]  # Example hall layout

    occupied_seats = set()
    if os.path.exists(attendance_file):
        attendance_data = pd.read_csv(attendance_file)
        for _, row in attendance_data.iterrows():
            occupied_seats.add((int(row['Exam Hall'].split('-')[1]), int(row['Seat ID'].split('-')[1])))

    # Remove already occupied seats
    free_seats = [seat for seat in available_seats if seat not in occupied_seats]

    # Group students from the same course together
    if course:
        free_seats = sorted(free_seats, key=lambda seat: seat[0])  # Group-based allocation
    else:
        np.random.shuffle(free_seats)  # Randomized allocation to prevent cheating

    if not free_seats:
        return "No seats available!"

    exam_hall, seat_id = free_seats[0]

    # Save attendance record
    attendance_status = "Present"
    data = {"Student ID": student_id, "Status": attendance_status, "Exam Hall": f"R-{exam_hall}", "Seat ID": f"S-{seat_id}"}
    df = pd.DataFrame([data])
    if os.path.exists(attendance_file):
        df.to_csv(attendance_file, mode='a', header=False, index=False)
    else:
        df.to_csv(attendance_file, mode='w', header=True, index=False)

    return f"Seat Allocated: R-{exam_hall} - S-{seat_id} | Attendance Marked: {attendance_status}"

# Real-Time Attendance Tracking
def track_attendance(video_source=0):
    cap = cv2.VideoCapture(video_source)
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        faces = detect_faces(frame)
        for face in faces:
            x, y, w, h = face
            face_img = frame[y:y+h, x:x+w]  # Crop face

            try:
                recognized_id = authenticate_student_from_frame(face_img)
                print(allocate_seat(recognized_id))
                cv2.putText(frame, f"ID: {recognized_id}", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            except:
                print("Unknown face detected")
                cv2.putText(frame, "Unknown", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

        cv2.imshow("Exam Attendance System", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# Face Detection using MTCNN
def detect_faces(frame):
    faces = detector.detect_faces(frame)
    detected_faces = []
    for face in faces:
        x, y, w, h = face['box']
        detected_faces.append((x, y, w, h))
    return detected_faces

# Authenticate Directly from Frame (captured stream)
def authenticate_student_from_frame(face_img, confidence_threshold=0.6):
    """Authenticates a student frame with security confidence threshold check."""
    if not hasattr(svm_model, "support_"):
        print("Authentication error: SVM model is not trained.")
        return None
    face_img_rgb = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
    face_img_resized = cv2.resize(face_img_rgb, (160, 160))  # FaceNet input size
    embeddings = facenet_model.embeddings([face_img_resized])
    if hasattr(svm_model, "predict_proba"):
        probs = svm_model.predict_proba(embeddings)[0]
        max_prob = np.max(probs)
        best_class = np.argmax(probs)
        if max_prob < confidence_threshold:
            print(f"Authentication failed: Low confidence score ({max_prob:.2f} < {confidence_threshold}).")
            return None
        return label_encoder.inverse_transform([best_class])[0]
    else:
        prediction = svm_model.predict(embeddings)
        return label_encoder.inverse_transform(prediction)[0]
# Example Usage
if __name__ == "__main__":
    print("Enhanced Face Recognition Exam System Loaded and Ready.")

Enhanced Face Recognition Exam System Loaded and Ready.


In [13]:
enroll_students(["/work/Student_img/student1.jpg", "/work/Student_img/student2.jpg"], ["ID_001", "ID_002"])
track_attendance("/work/exam_hall_video.mp4")
predicted_id = authenticate_student("/work/Student_img/student1.jpg")
print(allocate_seat(predicted_id))
track_attendance("/work/exam_hall_video.mp4")

1/1 [==============================] - 0s 99ms/step
Student data enrolled successfully!
1/1 [==============================] - 0s 132ms/step
Seat Allocated: R-1 - S-24 | Attendance Marked: Present


In [17]:
!pip install ultralytics==8.3.79

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 921.8/921.8 kB 55.3 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [54]:
import cv2
import numpy as np
import os
import joblib
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from keras.models import load_model
from keras.preprocessing.image import img_to_array
from keras.applications.vgg16 import preprocess_input
from mtcnn import MTCNN
from keras_facenet import FaceNet

# Load pre-trained FaceNet model for high-accuracy face recognition
facenet_model = FaceNet()

# Load existing models or initialize new ones
svm_model_path = "/work/svm_model.pkl"
label_encoder_path = "/work/label_encoder.pkl"
attendance_file = "/work/exam_attendance.csv"

if os.path.exists(svm_model_path) and os.path.exists(label_encoder_path):
    svm_model = joblib.load(svm_model_path)
    label_encoder = joblib.load(label_encoder_path)
else:
    svm_model = SVC(kernel='linear', probability=True)
    label_encoder = LabelEncoder()

detector = MTCNN()

# Helper function: Preprocess Image
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Error: Image file not found at {image_path}")
    
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return image  # FaceNet expects RGB images directly

# Feature Extraction using FaceNet
def extract_features(image_path):
    processed_image = preprocess_image(image_path)
    embeddings = facenet_model.embeddings([processed_image])
    return embeddings[0]

# Enrollment Phase (Train & Save Model)
def enroll_students(student_images, student_ids):
    features = []
    for image_path in student_images:
        features.append(extract_features(image_path))
    features = np.array(features)
    
    if len(student_ids) > 0:
        labels = label_encoder.fit_transform(student_ids)
        svm_model.fit(features, labels)
        joblib.dump(svm_model, svm_model_path)
        joblib.dump(label_encoder, label_encoder_path)
        print("Student data enrolled successfully!")

# Authentication Phase
def authenticate_student(image_path, confidence_threshold=0.6):
    """Authenticates a student image with security confidence threshold check."""
    if not hasattr(svm_model, "support_"):
        print("Authentication error: SVM model is not trained.")
        return None
    features = extract_features(image_path).reshape(1, -1)
    if hasattr(svm_model, "predict_proba"):
        probs = svm_model.predict_proba(features)[0]
        max_prob = np.max(probs)
        best_class = np.argmax(probs)
        if max_prob < confidence_threshold:
            print(f"Authentication failed: Low confidence score ({max_prob:.2f} < {confidence_threshold}).")
            return None
        return label_encoder.inverse_transform([best_class])[0]
    else:
        prediction = svm_model.predict(features)
        return label_encoder.inverse_transform(prediction)[0]

def allocate_seat(student_id, course=None):
    available_seats = [(r, s) for r in range(1, 6) for s in range(1, 31)]  # Example hall layout

    occupied_seats = set()
    if os.path.exists(attendance_file):
        attendance_data = pd.read_csv(attendance_file)
        for _, row in attendance_data.iterrows():
            occupied_seats.add((int(row['Exam Hall'].split('-')[1]), int(row['Seat ID'].split('-')[1])))

    # Remove already occupied seats
    free_seats = [seat for seat in available_seats if seat not in occupied_seats]

    # Group students from the same course together
    if course:
        free_seats = sorted(free_seats, key=lambda seat: seat[0])  # Group-based allocation
    else:
        np.random.shuffle(free_seats)  # Randomized allocation to prevent cheating

    if not free_seats:
        return "No seats available!"

    exam_hall, seat_id = free_seats[0]

    # Save attendance record
    attendance_status = "Present"
    data = {"Student ID": student_id, "Status": attendance_status, "Exam Hall": f"R-{exam_hall}", "Seat ID": f"S-{seat_id}"}
    df = pd.DataFrame([data])
    if os.path.exists(attendance_file):
        df.to_csv(attendance_file, mode='a', header=False, index=False)
    else:
        df.to_csv(attendance_file, mode='w', header=True, index=False)

    return f"Seat Allocated: R-{exam_hall} - S-{seat_id} | Attendance Marked: {attendance_status}"

# Real-Time Attendance Tracking
def track_attendance(video_source=0):
    cap = cv2.VideoCapture(video_source)
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        faces = detect_faces(frame)
        for face in faces:
            x, y, w, h = face
            face_img = frame[y:y+h, x:x+w]  # Crop face

            try:
                recognized_id = authenticate_student_from_frame(face_img)
                print(allocate_seat(recognized_id))
                cv2.putText(frame, f"ID: {recognized_id}", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            except:
                print("Unknown face detected")
                cv2.putText(frame, "Unknown", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

        cv2.imshow("Exam Attendance System", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# Face Detection using MTCNN
def detect_faces(frame):
    faces = detector.detect_faces(frame)
    detected_faces = []
    for face in faces:
        x, y, w, h = face['box']
        detected_faces.append((x, y, w, h))
    return detected_faces

# Authenticate Directly from Frame (captured stream)
def authenticate_student_from_frame(face_img, confidence_threshold=0.6):
    """Authenticates a student frame with security confidence threshold check."""
    if not hasattr(svm_model, "support_"):
        print("Authentication error: SVM model is not trained.")
        return None
    face_img_rgb = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
    face_img_resized = cv2.resize(face_img_rgb, (160, 160))  # FaceNet input size
    embeddings = facenet_model.embeddings([face_img_resized])
    if hasattr(svm_model, "predict_proba"):
        probs = svm_model.predict_proba(embeddings)[0]
        max_prob = np.max(probs)
        best_class = np.argmax(probs)
        if max_prob < confidence_threshold:
            print(f"Authentication failed: Low confidence score ({max_prob:.2f} < {confidence_threshold}).")
            return None
        return label_encoder.inverse_transform([best_class])[0]
    else:
        prediction = svm_model.predict(embeddings)
        return label_encoder.inverse_transform(prediction)[0]
# Example Usage
if __name__ == "__main__":
    print("Enhanced Face Recognition Exam System Loaded and Ready.")

Enhanced Face Recognition Exam System Loaded and Ready.


In [56]:
enroll_students(["/work/Student_img/student1.jpg", "/work/Student_img/student2.jpg"], ["ID_001", "ID_002"])
track_attendance("/work/exam_hall_video.mp4")
predicted_id = authenticate_student("/work/Student_img/student2.jpg")
print(allocate_seat(predicted_id))
track_attendance("/work/exam_hall_video.mp4")

1/1 [==============================] - 0s 97ms/step
Student data enrolled successfully!
1/1 [==============================] - 0s 94ms/step
Seat Allocated: R-3 - S-20 | Attendance Marked: Present


In [58]:
import cv2
import numpy as np
import os
import joblib
import pandas as pd
from datetime import datetime
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from keras.models import load_model
from keras.preprocessing.image import img_to_array
from keras.applications.vgg16 import preprocess_input
from mtcnn import MTCNN
from keras_facenet import FaceNet

# Load pre-trained FaceNet model for high-accuracy face recognition
facenet_model = FaceNet()

# Load existing models or initialize new ones
svm_model_path = "/work/svm_model.pkl"
label_encoder_path = "/work/label_encoder.pkl"
attendance_file = "/work/exam_attendance.csv"
unauthorized_log_file = "/work/unauthorized_log.csv"
mask_detection_file = "/work/mask_detection_log.csv"

if os.path.exists(svm_model_path) and os.path.exists(label_encoder_path):
    svm_model = joblib.load(svm_model_path)
    label_encoder = joblib.load(label_encoder_path)
else:
    svm_model = SVC(kernel='linear', probability=True)
    label_encoder = LabelEncoder()

detector = MTCNN()

# Helper function: Preprocess Image
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Error: Image file not found at {image_path}")
    
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return image  # FaceNet expects RGB images directly

# Feature Extraction using FaceNet
def extract_features(image_path):
    processed_image = preprocess_image(image_path)
    embeddings = facenet_model.embeddings([processed_image])
    return embeddings[0]

# Enrollment Phase (Train & Save Model)
def enroll_students(student_images, student_ids):
    features = []
    for image_path in student_images:
        features.append(extract_features(image_path))
    features = np.array(features)
    
    if len(student_ids) > 0:
        labels = label_encoder.fit_transform(student_ids)
        svm_model.fit(features, labels)
        joblib.dump(svm_model, svm_model_path)
        joblib.dump(label_encoder, label_encoder_path)
        print("Student data enrolled successfully!")

# Authentication Phase
def authenticate_student(image_path, confidence_threshold=0.6):
    """Authenticates a student image with security confidence threshold check."""
    if not hasattr(svm_model, "support_"):
        print("Authentication error: SVM model is not trained.")
        return None
    features = extract_features(image_path).reshape(1, -1)
    if hasattr(svm_model, "predict_proba"):
        probs = svm_model.predict_proba(features)[0]
        max_prob = np.max(probs)
        best_class = np.argmax(probs)
        if max_prob < confidence_threshold:
            print(f"Authentication failed: Low confidence score ({max_prob:.2f} < {confidence_threshold}).")
            return None
        return label_encoder.inverse_transform([best_class])[0]
    else:
        prediction = svm_model.predict(features)
        return label_encoder.inverse_transform(prediction)[0]

def log_unauthorized_student(face_img):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    unauthorized_image_path = f"/work/unauthorized_faces/{timestamp}.jpg"
    cv2.imwrite(unauthorized_image_path, face_img)
    data = {"Timestamp": timestamp, "Image Path": unauthorized_image_path}
    df = pd.DataFrame([data])
    if os.path.exists(unauthorized_log_file):
        df.to_csv(unauthorized_log_file, mode='a', header=False, index=False)
    else:
        df.to_csv(unauthorized_log_file, mode='w', header=True, index=False)

# Add mask detection logging (Placeholder logic for detecting masks)
def detect_and_log_mask(face_img):
    # Placeholder: Replace with Mask Detection Model/Logic
    mask_detected = True if np.random.rand() > 0.5 else False
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    mask_status = "Mask" if mask_detected else "No Mask"

    data = {"Timestamp": timestamp, "Mask Status": mask_status}
    df = pd.DataFrame([data])
    if os.path.exists(mask_detection_file):
        df.to_csv(mask_detection_file, mode='a', header=False, index=False)
    else:
        df.to_csv(mask_detection_file, mode='w', header=True, index=False)
    return mask_status

# Allocate Seat Dynamically 
def allocate_seat(student_id, course=None):
    available_seats = [(r, s) for r in range(1, 6) for s in range(1, 31)]  # Example hall layout

    occupied_seats = set()
    if os.path.exists(attendance_file):
        attendance_data = pd.read_csv(attendance_file)
        for _, row in attendance_data.iterrows():
            occupied_seats.add((int(row['Exam Hall'].split('-')[1]), int(row['Seat ID'].split('-')[1])))

    # Remove already occupied seats
    free_seats = [seat for seat in available_seats if seat not in occupied_seats]

    # Group students from the same course together
    if course:
        free_seats = sorted(free_seats, key=lambda seat: seat[0])  # Group-based allocation
    else:
        np.random.shuffle(free_seats)  # Randomized allocation to prevent cheating

    if not free_seats:
        return "No seats available!"

    exam_hall, seat_id = free_seats[0]

    # Save attendance record
    attendance_status = "Present"
    data = {"Student ID": student_id, "Status": attendance_status, "Exam Hall": f"R-{exam_hall}", "Seat ID": f"S-{seat_id}"}
    df = pd.DataFrame([data])
    if os.path.exists(attendance_file):
        df.to_csv(attendance_file, mode='a', header=False, index=False)
    else:
        df.to_csv(attendance_file, mode='w', header=True, index=False)

    return f"Seat Allocated: R-{exam_hall} - S-{seat_id} | Attendance Marked: {attendance_status}"

# Real-Time Attendance Tracking
def track_attendance(video_source=0):
    cap = cv2.VideoCapture(video_source)
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        faces = detect_faces(frame)
        for face in faces:
            x, y, w, h = face
            face_img = frame[y:y+h, x:x+w]  # Crop face

            try:
                recognized_id = authenticate_student_from_frame(face_img)
                mask_status = detect_and_log_mask(face_img)  # Detect mask and log
                print(f"{allocate_seat(recognized_id)} | Mask Status: {mask_status}")
                cv2.putText(frame, f"ID: {recognized_id} ({mask_status})", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            except:
                log_unauthorized_student(face_img)  # Log unauthorized student
                print("Unknown face detected - Logged")
                cv2.putText(frame, "Unknown", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

        cv2.imshow("Exam Attendance System", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# Face Detection using MTCNN
def detect_faces(frame):
    faces = detector.detect_faces(frame)
    detected_faces = []
    for face in faces:
        x, y, w, h = face['box']
        detected_faces.append((x, y, w, h))
    return detected_faces

# Authenticate Directly from Frame (captured stream)
def authenticate_student_from_frame(face_img, confidence_threshold=0.6):
    """Authenticates a student frame with security confidence threshold check."""
    if not hasattr(svm_model, "support_"):
        print("Authentication error: SVM model is not trained.")
        return None
    face_img_rgb = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
    face_img_resized = cv2.resize(face_img_rgb, (160, 160))  # FaceNet input size
    embeddings = facenet_model.embeddings([face_img_resized])
    if hasattr(svm_model, "predict_proba"):
        probs = svm_model.predict_proba(embeddings)[0]
        max_prob = np.max(probs)
        best_class = np.argmax(probs)
        if max_prob < confidence_threshold:
            print(f"Authentication failed: Low confidence score ({max_prob:.2f} < {confidence_threshold}).")
            return None
        return label_encoder.inverse_transform([best_class])[0]
    else:
        prediction = svm_model.predict(embeddings)
        return label_encoder.inverse_transform(prediction)[0]
# Example Usage
if __name__ == "__main__":
    print("Enhanced Face Recognition Exam System Loaded and Ready.")

Enhanced Face Recognition Exam System Loaded and Ready.


In [62]:
enroll_students(["/work/Student_img/student1.jpg", "/work/Student_img/student2.jpg", "/work/Student_img/student3.jpg"], ["ID_001", "ID_002","ID_003"])
track_attendance("/work/exam_hall_video.mp4")
predicted_id = authenticate_student("/work/Student_img/student3.jpg")
print(allocate_seat(predicted_id))
track_attendance("/work/exam_hall_video.mp4")

# enroll_students([
#     "/work/Student_img/student1.jpg",
#     "/work/Student_img/student2.jpg",
#     "/work/Student_img/student3.jpg"
# ], ["ID_001", "ID_002", "ID_003"])

# predicted_ids = authenticate_students(["/work/Student_img/student1.jpg"])
# print("Authenticated IDs:", predicted_ids)
# track_attendance_with_yolo(video_source="/work/exam_hall_video.mp4")

1/1 [==============================] - 0s 112ms/step
Student data enrolled successfully!
1/1 [==============================] - 0s 118ms/step
Seat Allocated: R-1 - S-30 | Attendance Marked: Present


In [7]:
!pip install keras_facenet==0.3.2

  Preparing metadata (setup.py) ... done
  Created wheel for keras_facenet: filename=keras_facenet-0.3.2-py3-none-any.whl size=10368 sha256=b6c38b3f217bfbf89d263e34486d50bf4384d4ff5598f3c0b05f81995f865749
  Stored in directory: /root/.cache/pip/wheels/1d/d8/a9/85cf04ea29321d2afcb82c0caaafdca9195385f9d68cbc7185
Successfully built keras_facenet

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import cv2
import numpy as np
import os
import joblib
import pandas as pd
from datetime import datetime
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from keras.models import load_model
from keras.preprocessing.image import img_to_array
from keras.applications.vgg16 import preprocess_input
from mtcnn import MTCNN
from keras_facenet import FaceNet

facenet_model = FaceNet()
mask_detector_model_path = '/work/mask_detector.model'
mask_detector_model = load_model(mask_detector_model_path)
# print("Mask Detection Model Loaded!")
# print("Model Summary:", model.summary())

svm_model_path = "/work/svm_model.pkl"
label_encoder_path = "/work/label_encoder.pkl"
attendance_file = "/work/exam_attendance.csv"
unauthorized_log_file = "/work/unauthorized_log.csv"
mask_detection_file = "/work/mask_detection_log.csv"

if os.path.exists(svm_model_path) and os.path.exists(label_encoder_path):
    svm_model = joblib.load(svm_model_path)
    label_encoder = joblib.load(label_encoder_path)
else:
    svm_model = SVC(kernel='linear', probability=True)
    label_encoder = LabelEncoder()

detector = MTCNN()

# Helper function: Preprocess Image
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Error: Image file not found at {image_path}")
    
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return image  # FaceNet expects RGB images directly

# Feature Extraction using FaceNet
def extract_features(image_path):
    processed_image = preprocess_image(image_path)
    embeddings = facenet_model.embeddings([processed_image])
    return embeddings[0]

# Enrollment Phase (Train & Save Model)
def enroll_students(student_images, student_ids):
    features = []
    for image_path in student_images:
        features.append(extract_features(image_path))
    features = np.array(features)
    
    if len(student_ids) > 0:
        labels = label_encoder.fit_transform(student_ids)
        svm_model.fit(features, labels)
        joblib.dump(svm_model, svm_model_path)
        joblib.dump(label_encoder, label_encoder_path)
        print("Student data enrolled successfully!")

def authenticate_student(image_path, confidence_threshold=0.6):
    """Authenticates a student image with security confidence threshold check."""
    if not hasattr(svm_model, "support_"):
        print("Authentication error: SVM model is not trained.")
        return None
    features = extract_features(image_path).reshape(1, -1)
    if hasattr(svm_model, "predict_proba"):
        probs = svm_model.predict_proba(features)[0]
        max_prob = np.max(probs)
        best_class = np.argmax(probs)
        if max_prob < confidence_threshold:
            print(f"Authentication failed: Low confidence score ({max_prob:.2f} < {confidence_threshold}).")
            return None
        return label_encoder.inverse_transform([best_class])[0]
    else:
        prediction = svm_model.predict(features)
        return label_encoder.inverse_transform(prediction)[0]

def log_unauthorized_student(face_img):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    unauthorized_image_path = f"/work/unauthorized_faces/{timestamp}.jpg"
    os.makedirs(os.path.dirname(unauthorized_image_path), exist_ok=True)
    cv2.imwrite(unauthorized_image_path, face_img)
    data = {"Timestamp": timestamp, "Image Path": unauthorized_image_path}
    df = pd.DataFrame([data])
    if os.path.exists(unauthorized_log_file):
        df.to_csv(unauthorized_log_file, mode='a', header=False, index=False)
    else:
        df.to_csv(unauthorized_log_file, mode='w', header=True, index=False)

# Detect and Log Mask Information using Pre-Trained Model
def detect_and_log_mask(face_img):
    face_img = cv2.resize(face_img, (224, 224))  # Resize for MobileNetV2
    face_img = img_to_array(face_img)
    face_img = preprocess_input(face_img)
    face_img = np.expand_dims(face_img, axis=0)  # Expand dims for model input

    # Predict mask or no mask
    mask_prob = mask_detector_model.predict(face_img)[0][0]
    print(f"DEBUG: Mask Probability: {mask_prob}")  # Debugging Line

    mask_detected = mask_prob > 0.3  # If probability > 50%, mask is present
    mask_status = "Mask" if mask_detected else "No Mask"

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    data = {"Timestamp": timestamp, "Mask Status": mask_status}
    df = pd.DataFrame([data])
    if os.path.exists(mask_detection_file):
        df.to_csv(mask_detection_file, mode='a', header=False, index=False)
    else:
        df.to_csv(mask_detection_file, mode='w', header=True, index=False)

    return mask_status, mask_detected



# Allocate Seat Dynamically
def allocate_seat(student_id, course=None):
    available_seats = [(r, s) for r in range(1, 6) for s in range(1, 31)]  # Example hall layout

    occupied_seats = set()
    if os.path.exists(attendance_file):
        attendance_data = pd.read_csv(attendance_file)
        for _, row in attendance_data.iterrows():
            occupied_seats.add((int(row['Exam Hall'].split('-')[1]), int(row['Seat ID'].split('-')[1])))

    # Remove already occupied seats
    free_seats = [seat for seat in available_seats if seat not in occupied_seats]

    # Group students from the same course together
    if course:
        free_seats = sorted(free_seats, key=lambda seat: seat[0])  # Group-based allocation
    else:
        np.random.shuffle(free_seats)  # Randomized allocation to prevent cheating

    if not free_seats:
        return "No seats available!"

    exam_hall, seat_id = free_seats[0]

    # Save attendance record
    attendance_status = "Present"
    data = {"Student ID": student_id, "Status": attendance_status, "Exam Hall": f"R-{exam_hall}", "Seat ID": f"S-{seat_id}"}
    df = pd.DataFrame([data])
    if os.path.exists(attendance_file):
        df.to_csv(attendance_file, mode='a', header=False, index=False)
    else:
        df.to_csv(attendance_file, mode='w', header=True, index=False)

    return f"Seat Allocated: R-{exam_hall} - S-{seat_id} | Attendance Marked: {attendance_status}"

# Real-Time Attendance Tracking
def track_attendance(image_path):
    frame = cv2.imread(image_path)

    if frame is None:
        print("Error: Unable to read the image file.")
        return

    faces = detect_faces(frame)
    for face in faces:
        x, y, w, h = face
        face_img = frame[y:y+h, x:x+w]  # Crop face

        # Detect mask first
        mask_status, is_mask_detected = detect_and_log_mask(face_img)

        if is_mask_detected:
            print(f"Mask detected! Student cannot be authenticated.")
            cv2.putText(frame, "Mask Detected! Remove mask", (x, y - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            continue  # Skip authentication if mask is detected

        student_images, student_ids = extract_faces(face_img)

        if len(student_images) == 0:
            print("No face detected in the image!")
            cv2.putText(frame, "No face detected in the image!", (x, y - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            continue

        if len(student_ids) == 0:
            print("No ID detected in the image!")
            cv2.putText(frame, "No ID detected in the image!", (x, y - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            continue

        if not os.path.exists(svm_model_path) or not os.path.exists(label_encoder_path):
            enroll_students(student_images, student_ids)

        if enroll_students:
            enroll_students(student_images, student_ids)

        if not os.path.exists(attendance_file):
            df = pd.DataFrame(columns=["Student ID", "Status", "Exam Hall", "Seat ID"])
            df.to_csv(attendance_file, mode='w', header=True, index=False)

        if not os.path.exists(unauthorized_log_file):
            df = pd.DataFrame(columns=["Timestamp", "Image Path"])
            df.to_csv(unauthorized_log_file, mode='w', header=True, index=False)

        if not os.path.exists(mask_detection_file):
            df = pd.DataFrame(columns=["Timestamp", "Mask Status"])
            df.to_csv(mask_detection_file, mode='w', header=True, index=False)



        try:
            recognized_id = authenticate_student_from_frame(face_img)
            seat_info = allocate_seat(recognized_id)
            print(f"{seat_info} | Mask Status: {mask_status}")
            cv2.putText(frame, f"ID: {recognized_id} ({mask_status})", (x, y - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        except Exception as e:
            log_unauthorized_student(face_img)  # Log unauthorized student
            print(f"Unauthorized face detected or error encountered: {str(e)}")
            cv2.putText(frame, "Unauthorized!", (x, y - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    cv2.imshow("Exam Attendance System", frame)
    cv2.waitKey(0)  # Wait for a key press
    cv2.destroyAllWindows()


# Face Detection using MTCNN
def detect_faces(frame):
    faces = detector.detect_faces(frame)
    detected_faces = []
    for face in faces:
        if face['confidence'] > 0.9:
            x, y, w, h = face['box']
            detected_faces.append((x, y, w, h))
    return detected_faces

# Authenticate Directly from Frame (captured stream)
def authenticate_student_from_frame(face_img, confidence_threshold=0.6):
    """Authenticates a student frame with security confidence threshold check."""
    if not hasattr(svm_model, "support_"):
        print("Authentication error: SVM model is not trained.")
        return None
    face_img_rgb = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
    face_img_resized = cv2.resize(face_img_rgb, (160, 160))  # FaceNet input size
    embeddings = facenet_model.embeddings([face_img_resized])
    if hasattr(svm_model, "predict_proba"):
        probs = svm_model.predict_proba(embeddings)[0]
        max_prob = np.max(probs)
        best_class = np.argmax(probs)
        if max_prob < confidence_threshold:
            print(f"Authentication failed: Low confidence score ({max_prob:.2f} < {confidence_threshold}).")
            return None
        return label_encoder.inverse_transform([best_class])[0]
    else:
        prediction = svm_model.predict(embeddings)
        return label_encoder.inverse_transform(prediction)[0]
# Example Usage
if __name__ == "__main__":
    print("Enhanced Face Recognition Exam System Loaded and Ready.")

2025-02-27 04:55:55.980393: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-27 04:55:55.983189: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-02-27 04:55:56.064413: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-02-27 04:55:56.064549: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-02-27 04:55:56.065608: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

In [2]:
# image_path = "/work/Student_img/student1.jpg"
# recognized_id = authenticate_student(image_path)
# print(f"Recognized Student ID: {recognized_id}")

# seat_info = allocate_seat(recognized_id)
# print(seat_info)

track_attendance("/work/Student_img/student2.jpg")

NameError: name 'track_attendance' is not defined

In [41]:
prediction = mask_detector_model.predict(face_img)
print(f"DEBUG: Prediction Shape: {prediction.shape}, Values: {prediction}")

NameError: name 'face_img' is not defined

In [47]:
# enroll_students(["/work/Student_img/student1.jpg", "/work/Student_img/student2.jpg", "/work/Student_img/student3.jpg"], ["ID_001", "ID_002","ID_003"])
# track_attendance("/work/exam_hall_video.mp4")
# predicted_id = authenticate_student("/work/Student_img/student3.jpg")
# print(allocate_seat(predicted_id))
# track_attendance("/work/exam_hall_video.mp4")
# image = cv2.imread("/work/Student_img/student2.jpg")
# mask_status, is_mask_detected = detect_and_log_mask(image)
# print(f"Mask Status: {mask_status}")
student_id = authenticate_student("/work/Student_img/student1.jpg")
print(f"Recognized Student ID: {student_id}")

# enroll_students([
#     "/work/Student_img/student1.jpg",
#     "/work/Student_img/student2.jpg",
#     "/work/Student_img/student3.jpg"
# ], ["ID_001", "ID_002", "ID_003"])

# predicted_ids = authenticate_students(["/work/Student_img/student1.jpg"])
# print("Authenticated IDs:", predicted_ids)
# track_attendance_with_yolo(video_source="/work/exam_hall_video.mp4")

1/1 [==============================] - 1s 1s/step
Recognized Student ID: ID_001


In [23]:
import cv2
import numpy as np
import os
import joblib
import pandas as pd
from datetime import datetime
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from keras.models import load_model
from keras.preprocessing.image import img_to_array
from keras.applications.vgg16 import preprocess_input
from mtcnn import MTCNN
from keras_facenet import FaceNet

facenet_model = FaceNet()
mask_detector_model_path = '/work/mask_detector.model'
mask_detector_model = load_model(mask_detector_model_path)
svm_model_path = "/work/svm_model.pkl"
label_encoder_path = "/work/label_encoder.pkl"
attendance_file = "/work/Attendance/exam_attendance.csv"

if os.path.exists(svm_model_path) and os.path.exists(label_encoder_path):
    svm_model = joblib.load(svm_model_path)
    label_encoder = joblib.load(label_encoder_path)
else:
    svm_model = SVC(kernel='linear', probability=True)
    label_encoder = LabelEncoder()

detector = MTCNN()

def preprocess_image(image_path):
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Error: Image file not found at {image_path}")
    
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return image

def extract_features(image_path):
    processed_image = preprocess_image(image_path)
    embeddings = facenet_model.embeddings([processed_image])
    return embeddings[0]

def enroll_students(student_images, student_ids):
    features = []
    for image_path in student_images:
        features.append(extract_features(image_path))
    features = np.array(features)

    if len(student_ids) > 0:
        labels = label_encoder.fit_transform(student_ids)
        svm_model.fit(features, labels)
        joblib.dump(svm_model, svm_model_path)
        joblib.dump(label_encoder, label_encoder_path)
        print("Student data enrolled successfully!")

def authenticate_student(image_path, confidence_threshold=0.6):
    """Authenticates a student image with security confidence threshold check."""
    if not hasattr(svm_model, "support_"):
        print("Authentication error: SVM model is not trained.")
        return None
    features = extract_features(image_path).reshape(1, -1)
    if hasattr(svm_model, "predict_proba"):
        probs = svm_model.predict_proba(features)[0]
        max_prob = np.max(probs)
        best_class = np.argmax(probs)
        if max_prob < confidence_threshold:
            print(f"Authentication failed: Low confidence score ({max_prob:.2f} < {confidence_threshold}).")
            return None
        return label_encoder.inverse_transform([best_class])[0]
    else:
        prediction = svm_model.predict(features)
        return label_encoder.inverse_transform(prediction)[0]

def detect_faces(frame):
    faces = detector.detect_faces(frame)
    detected_faces = []
    for face in faces:
        if face['confidence'] > 0.9:
            x, y, w, h = face['box']
            detected_faces.append((x, y, w, h))
    return detected_faces


def detect_and_log_mask(face_img):
    face_img = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
    face_img = cv2.resize(face_img, (224, 224))
    face_img = img_to_array(face_img)
    face_img = preprocess_input(face_img)
    face_img = np.expand_dims(face_img, axis=0)

    mask_prob = mask_detector_model.predict(face_img)[0][0]
    # print(f"DEBUG: Mask Probability: {mask_prob}")

    mask_detected = mask_prob > 0.5  
    mask_status = "Mask" if mask_detected else "No Mask"

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    data = {"Timestamp": timestamp, "Mask Status": mask_status}
    df = pd.DataFrame([data])
    if os.path.exists(mask_detection_file):
        df.to_csv(mask_detection_file, mode='a', header=False, index=False)
    else:
        df.to_csv(mask_detection_file, mode='w', header=True, index=False)

    return mask_status, mask_detected

def get_occupied_seats():
    occupied_seats = set()
    if os.path.exists(attendance_file):
        attendance_data = pd.read_csv(attendance_file)
        for _, row in attendance_data.iterrows():
            occupied_seats.add((int(row['Exam Hall'].split('-')[1]), int(row['Seat ID'].split('-')[1])))
    return occupied_seats

def allocate_seat(course=None):
    available_seats = [(r, s) for r in range(1, 6) for s in range(1, 31)]  # Example hall layout
    occupied_seats = get_occupied_seats()
    free_seats = [seat for seat in available_seats if seat not in occupied_seats]

    if not free_seats:
        return None
    if course:
        free_seats = sorted(free_seats, key=lambda seat: seat[0])  
    else:
        np.random.shuffle(free_seats)

    return free_seats[0]

def mark_attendance(student_id, seat):
    if not seat:
        return "No seats available!"

    exam_hall, seat_id = seat
    attendance_status = "Present"
    data = {
        "Student ID": student_id,
        "Status": attendance_status,
        "Exam Hall": f"R-{exam_hall}",
        "Seat ID": f"S-{seat_id}"
    }

    df = pd.DataFrame([data])
    if os.path.exists(attendance_file):
        df.to_csv(attendance_file, mode='a', header=False, index=False)
    else:
        df.to_csv(attendance_file, mode='w', header=True, index=False)

    return f"Seat Allocated: R-{exam_hall} - S-{seat_id} | Attendance Marked: {attendance_status}"


# Authenticate Directly from Frame (captured stream)
def authenticate_student_from_frame(face_img, confidence_threshold=0.6):
    """Authenticates a student frame with security confidence threshold check."""
    if not hasattr(svm_model, "support_"):
        print("Authentication error: SVM model is not trained.")
        return None
    face_img_rgb = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
    face_img_resized = cv2.resize(face_img_rgb, (160, 160))  # FaceNet input size
    embeddings = facenet_model.embeddings([face_img_resized])
    if hasattr(svm_model, "predict_proba"):
        probs = svm_model.predict_proba(embeddings)[0]
        max_prob = np.max(probs)
        best_class = np.argmax(probs)
        if max_prob < confidence_threshold:
            print(f"Authentication failed: Low confidence score ({max_prob:.2f} < {confidence_threshold}).")
            return None
        return label_encoder.inverse_transform([best_class])[0]
    else:
        prediction = svm_model.predict(embeddings)
        return label_encoder.inverse_transform(prediction)[0]
def general_function(face_image, student_images=None, student_ids=None):
    try:
        # Step 0: Detect faces
        detected_faces = detect_faces(preprocess_image(face_image))
        if not detected_faces:
            print("No faces detected in the image!")
            return None

        print(f"Step 0: Number of faces detected: {len(detected_faces)}")

        # Step 1: Detect and log mask status
        processed_image = preprocess_image(face_image)
        mask_status, is_mask_detected = detect_and_log_mask(processed_image)
        print(f"Step 1: Mask Status: {mask_status}")

        if is_mask_detected:
            print(f"Mask detected! Student cannot be authenticated.")
            return None

        if student_images and student_ids:
            enroll_students(student_images, student_ids)

        recognized_id = authenticate_student(face_image)
        if recognized_id is None:
            print("Authentication failed: No matching student found.")
            return None

        seat_info = allocate_seat()
        if seat_info is None:
            print("No seats available!")
            return None


        mark_attendance(recognized_id, seat_info)

        print(f"Student ID: {recognized_id}, {seat_info}")
        return recognized_id, seat_info

    except FileNotFoundError as e:
        print(f"Error: {e}")
        return None
    except Exception as e:
        print(f"Unexpected error encountered: {e}")
        return None


if __name__ == "__main__":
    print("Face Recognition Exam System Loaded and Ready.")

Enhanced Face Recognition Exam System Loaded and Ready.


In [25]:
face_image = "/work/Student_img/student2.jpg"

result = general_function(face_image)

if result:
    recognized_id, seat_info = result
    print(f"✅ Recognized Student ID: {recognized_id}, {seat_info}")
else:
    print("❌ No student recognized or attendance failed.")


Step 0: Number of faces detected: 1
1/1 [==============================] - 1s 626ms/step
DEBUG: Mask Probability: 0.007527469657361507
Step 1: Mask Status: No Mask
1/1 [==============================] - 1s 1s/step
Student ID: ID_002, (2, 24)
✅ Output: Recognized Student ID: ID_002, (2, 24)


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=656a6e4b-42df-434c-8df0-b6590b48d22a' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>